<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 2 &middot; Week 4 &middot; Part 4 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Practice Lab: Neural Networks for Binary Classification

This is the lecture's handwritten-digit network, built for real. Each image is an $8 \times 8$
grid of pixels &mdash; 64 numbers &mdash; and the network decides whether it shows a **0** or a
**1**:

$$64 \;\to\; 25 \;\to\; 15 \;\to\; 1, \qquad \text{all sigmoid}, \qquad 2031 \text{ parameters}.$$

You will train it in TensorFlow, then take its trained weights and run forward propagation
yourself in NumPy &mdash; first layer by layer, then vectorised over the whole test set &mdash;
and show that your code and TensorFlow make the same predictions. If you can do that, there is
nothing inside `model.predict` you do not understand.

This notebook is the **assessed practice lab** for Week 4. The two checkpoints and the exercises
at the end are what you hand in.

**Companion theory lecture:** CSE 815, Week 4, Part 1 ("Handwritten digits: 0 or 1?") and
Part 2 (Neural Networks in Code).

## What you will be able to do

1. Load an image data set, flatten images into feature vectors, and check every shape.
2. Split data into training and test sets, and say why the test set is held back.
3. Build, train and evaluate a three-layer network in TensorFlow.
4. Extract a trained model's weights and reproduce its predictions with your own NumPy forward
   propagation &mdash; looped and vectorised.
5. Inspect what the first layer responds to, and which test images the model is least sure of.

---

## 1. Setup and the data

The images come with scikit-learn (`load_digits`), so nothing is downloaded. Each pixel is a
brightness from 0 (white) to 16 (black). We keep only the 0s and 1s, and divide by 16 so that
every feature lies in $[0, 1]$ &mdash; for images, that is all the feature scaling needed.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow's start-up chatter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)
keras.utils.set_random_seed(816)      # seeds Python, NumPy and TensorFlow together

BLUE, TEAL, AMBER, ROSE, GRAY = "#123B63", "#1B7F79", "#C97B17", "#9B2C4B", "#4A5560"

print("NumPy", np.__version__, "  TensorFlow", tf.__version__)

from sklearn.datasets import load_digits

digits = load_digits(n_class=2)           # only the digits 0 and 1
images = digits.images                    # (m, 8, 8)
X = digits.data / 16.0                    # (m, 64): each image flattened row by row
y = digits.target.astype(float)           # (m,): 0.0 or 1.0

print("images:", images.shape, "  X:", X.shape, "  y:", y.shape)
print(f"zeros: {int((y == 0).sum())}   ones: {int((y == 1).sum())}")
print(f"pixel range after scaling: {X.min():.1f} .. {X.max():.1f}")

assert X.shape == (360, 64) and y.shape == (360,)
assert np.array_equal(X[0], images[0].ravel() / 16.0)   # flattening is row by row

In [ ]:
rng = np.random.default_rng(816)
show = rng.choice(len(y), size=16, replace=False)

fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, i in zip(axes.ravel(), show):
    ax.imshow(images[i], cmap="gray_r")
    ax.set_title(f"y = {int(y[i])}", fontsize=9)
    ax.axis("off")
fig.suptitle("A random sample of the 360 images", fontsize=10)
plt.tight_layout()
plt.show()

### One image, as the network sees it

The network never sees a picture. It sees a row of 64 numbers. Print one:

In [ ]:
i = show[0]
print(f"example {i}, label {int(y[i])}, as an 8 x 8 grid (brightness 0..16):\n")
print(images[i].astype(int))
print(f"\n... and as the network receives it, one row of 64 features:\n{X[i]}")

---

## 2. Training and test sets

From this week on, we judge a model on data it was **not** trained on. Week 3, Part 4 showed
why: training accuracy can be made perfect by a model that has learned nothing general. We hold
back 80 images as a test set, and touch them only to evaluate.

In [ ]:
rng = np.random.default_rng(816)
idx = rng.permutation(len(y))
train, test = idx[:280], idx[280:]

X_train, y_train = X[train], y[train]
X_test, y_test = X[test], y[test]
print("X_train", X_train.shape, "  X_test", X_test.shape)
print(f"share of 1s -- train: {y_train.mean():.3f}   test: {y_test.mean():.3f}")

assert len(set(train) & set(test)) == 0 and len(train) + len(test) == len(y)

---

## 3. Build the network

Exactly the lecture's architecture. `Input(shape=(64,))` fixes the number of features; the three
`Dense` layers have 25, 15 and 1 units.

In [ ]:
keras.utils.set_random_seed(816)

model = Sequential([
    Input(shape=(64,)),
    Dense(units=25, activation="sigmoid", name="L1"),
    Dense(units=15, activation="sigmoid", name="L2"),
    Dense(units=1, activation="sigmoid", name="L3"),
], name="digits_0_vs_1")
model.summary()

assert [layer.count_params() for layer in model.layers] == [1625, 390, 16]
assert model.count_params() == 2031
print("Parameter counts match the lecture: 1625 + 390 + 16 = 2031.")

Check the shapes of the weights against the convention from Part 1: $\mathbf{W}$ is
`(n_in, units)`, $\mathbf{b}$ is `(units,)`.

In [ ]:
for layer in model.layers:
    W, b = layer.get_weights()
    print(f"{layer.name}:  W {str(W.shape):9}  b {str(b.shape):6}")

[W1, b1] = model.get_layer("L1").get_weights()
assert W1.shape == (64, 25) and b1.shape == (25,)

---

## 4. Train and evaluate

As in Part 2, `compile` and `fit` are Week 5's subject; today they are given. `BinaryCrossentropy`
is the logistic cost from Week 3.

In [ ]:
model.compile(
    loss=keras.losses.BinaryCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
)
history = model.fit(X_train, y_train, epochs=50, verbose=0)

loss = history.history["loss"]
for epoch in [1, 2, 5, 10, 20, 50]:
    print(f"epoch {epoch:2d}   cost {loss[epoch - 1]:.6f}")

fig, ax = plt.subplots(figsize=(6.0, 3.4))
ax.plot(np.arange(1, 51), loss, c=BLUE, lw=2)
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("training cost")
ax.set_title("Training cost, log scale", fontsize=10)
plt.show()

In [ ]:
def accuracy(model, X, y):
    return float(((model.predict(X, verbose=0)[:, 0] >= 0.5) == y).mean())


acc_train = accuracy(model, X_train, y_train)
acc_test = accuracy(model, X_test, y_test)
print(f"training accuracy: {acc_train:.4f}")
print(f"test accuracy    : {acc_test:.4f}")

assert acc_test >= 0.97, "unexpectedly poor -- re-run the build and train cells"

0 versus 1 is an easy pair: the reference run classifies every test image correctly. Exercise 1
asks you to try a harder one.

### Predicting one image: mind the shape

`model.predict` expects a batch, shape `(m, 64)`. A single image `X_test[0]` has shape `(64,)`.

In [ ]:
x0 = X_test[0]
print("X_test[0].shape     :", x0.shape)

try:
    model.predict(x0, verbose=0)
except Exception as exc:                      # the exact exception type varies by version
    print("predict(X_test[0])  ->", type(exc).__name__, "(wrong shape)")

p0 = model.predict(x0.reshape(1, -1), verbose=0)     # or X_test[0:1]
print("predict(x0.reshape(1, -1)) ->", p0, "  shape", p0.shape)
print(f"label {int(y_test[0])}, prediction {int(p0[0, 0] >= 0.5)}")

---

## 5. Forward propagation in NumPy, with the trained weights

Now take the library away. Pull out the six trained arrays and run the network yourself.

### 5.1 One layer at a time, with a loop

This is `my_dense` from Part 3: a loop over units, each unit a dot product, a bias and a sigmoid.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def my_dense(a_in, W, b):
    """a_in: (n_in,)   W: (n_in, units)   b: (units,)   ->   (units,)"""
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):
        a_out[j] = sigmoid(np.dot(W[:, j], a_in) + b[j])
    return a_out


[W1, b1] = model.get_layer("L1").get_weights()
[W2, b2] = model.get_layer("L2").get_weights()
[W3, b3] = model.get_layer("L3").get_weights()

x0 = X_test[0]
a1 = my_dense(x0, W1, b1)
a2 = my_dense(a1, W2, b2)
a3 = my_dense(a2, W3, b3)
print("shapes along the way:", x0.shape, "->", a1.shape, "->", a2.shape, "->", a3.shape)
print(f"my forward propagation: {a3[0]:.8f}")
print(f"model.predict         : {model.predict(X_test[0:1], verbose=0)[0, 0]:.8f}")

assert a1.shape == (25,) and a2.shape == (15,) and a3.shape == (1,)
assert np.isclose(a3[0], model.predict(X_test[0:1], verbose=0)[0, 0], atol=1e-6)

The two agree to about seven decimal places &mdash; the precision of float32, in which
TensorFlow stored the weights and computed its answer. Your NumPy code ran in float64.

### 5.2 The whole test set at once

The vectorised layer computes all units for all examples in one matrix product:
$\mathbf{A}_{\text{out}} = g(\mathbf{A}_{\text{in}}\mathbf{W} + \mathbf{b})$, with shapes
$(m, n_{\text{in}})(n_{\text{in}}, \text{units}) \to (m, \text{units})$.

In [ ]:
def my_dense_v(A_in, W, b):
    """A_in: (m, n_in)   W: (n_in, units)   b: (units,)   ->   (m, units)"""
    return sigmoid(A_in @ W + b)


A1 = my_dense_v(X_test, W1, b1)
A2 = my_dense_v(A1, W2, b2)
A3 = my_dense_v(A2, W3, b3)
print("shapes:", X_test.shape, "->", A1.shape, "->", A2.shape, "->", A3.shape)

P_tf = model.predict(X_test, verbose=0)
print(f"largest |mine - TensorFlow| over {len(y_test)} test images: "
      f"{np.abs(A3 - P_tf).max():.2e}")
assert A3.shape == P_tf.shape == (80, 1)
assert np.allclose(A3, P_tf, atol=1e-6)
assert np.array_equal(A3 >= 0.5, P_tf >= 0.5)

# And the loop agrees with the matrix product on every image.
A3_loop = np.array([my_dense(my_dense(my_dense(x, W1, b1), W2, b2), W3, b3) for x in X_test])
assert np.allclose(A3_loop, A3)
print("Loop, matrix product and TensorFlow: the same 80 predictions.")

---

## 6. Looking inside

### What does the first layer respond to?

Each first-layer unit has 64 weights, one per pixel. Reshape them to $8 \times 8$ and they form
an image of the pattern that most excites the unit: positive weights (blue) where ink raises its
activation, negative (red) where ink lowers it.

In [ ]:
fig, axes = plt.subplots(3, 9, figsize=(11, 4.0))
vmax = np.abs(W1).max()
for j, ax in enumerate(axes.ravel()):
    if j < W1.shape[1]:
        ax.imshow(W1[:, j].reshape(8, 8), cmap="RdBu", vmin=-vmax, vmax=vmax)
        ax.set_title(f"unit {j}", fontsize=8)
    ax.axis("off")
fig.suptitle("First-layer weights, one 8 x 8 image per hidden unit (blue +, red -)",
             fontsize=10)
plt.tight_layout()
plt.show()

# Compare each unit's weight image with the difference between the average 1 and the average 0.
mean_diff = X_train[y_train == 1].mean(axis=0) - X_train[y_train == 0].mean(axis=0)
corr = np.array([np.corrcoef(W1[:, j], mean_diff)[0, 1] for j in range(W1.shape[1])])
print("correlation of each unit's weights with (average 1 - average 0):")
print(np.round(corr, 2))
print(f"\nunits with |correlation| > 0.5: {int((np.abs(corr) > 0.5).sum())} of {len(corr)}")

In the reference run 22 of the 25 units have a weight image strongly correlated (positively or
negatively) with the difference between the average 1 and the average 0. For a pair this easy,
most of the first layer learns a version of the same template: the network has far more
capacity than the task needs. Expect more varied units on a harder pair.

In [ ]:
# How strongly does each unit's activation differ between the two classes?
H = my_dense_v(X_train, W1, b1)                       # (280, 25) first-layer activations
gap = H[y_train == 1].mean(axis=0) - H[y_train == 0].mean(axis=0)

order = np.argsort(-np.abs(gap))
print("first-layer units that best separate the classes (mean activation on 1s minus 0s):")
for j in order[:5]:
    print(f"  unit {j:2d}: {gap[j]:+.3f}")
print(f"\nunits with |gap| > 0.5: {int((np.abs(gap) > 0.5).sum())} of {len(gap)}")

### Which test images is the model least sure of?

Accuracy hides confidence. Sort the test images by how close their output is to the 0.5
threshold, and look at the hardest few.

In [ ]:
p = A3[:, 0]
closest = np.argsort(np.abs(p - 0.5))[:8]

fig, axes = plt.subplots(1, 8, figsize=(10, 1.9))
for ax, k in zip(axes, closest):
    ax.imshow(X_test[k].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"y={int(y_test[k])}  P={p[k]:.3f}", fontsize=8)
    ax.axis("off")
fig.suptitle("The eight test images closest to the decision threshold", fontsize=10)
plt.tight_layout()
plt.show()

print(f"smallest distance from 0.5 on the test set: {np.abs(p - 0.5).min():.4f}")

Even the least sure test images are classified with probabilities within a few hundredths of 0
or 1 in the reference run: on this pair the model is never close to the threshold. Checkpoint 2
shows a pair where it is.

### Checkpoint 1

Write `my_predict(X, params, threshold=0.5)` that takes any batch `X` of shape `(m, 64)` and a
list `params = [(W1, b1), (W2, b2), (W3, b3)]`, runs vectorised forward propagation, and returns
**0/1 predictions** of shape `(m,)`. Then:

1. Build `params` from `model.get_weights()` (it returns the six arrays in order).
2. Assert that `my_predict(X_test, params)` equals the thresholded `model.predict(X_test)`.
3. Report your accuracy on the test set, and how many of the 360 images in the **whole** data
   set your function classifies wrongly.

*Expected:* identical predictions; test accuracy `1.0`; `0` of the 360 images misclassified in
the reference run.

In [ ]:
# Your code here

### Checkpoint 2

0 versus 1 is easy. Repeat the whole pipeline for **8 versus 9**, two digits that share their
whole upper loop.

1. Load the full data set with `load_digits()`, keep the images whose target is 8 or 9, scale
   by 16, and set `y = 1` for a 9 and `y = 0` for an 8.
2. Split with `rng = np.random.default_rng(816)`, `idx = rng.permutation(m)`, using the first
   `int(0.78 * m)` indices for training and the rest for testing.
3. Build the same $64 \to 25 \to 15 \to 1$ network after `keras.utils.set_random_seed(816)`,
   train it exactly as in section 4, and report training and test accuracy.
4. Show the test images it gets wrong, if any, with their predicted probabilities.

*Expected (reference machine):* `354` images, `276` for training; training accuracy `1.0` and
test accuracy about `0.9487` &mdash; 4 of the 78 test images wrong. The test figure may differ by
an image or two on other platforms. Perfect training accuracy with imperfect test accuracy is a
pattern you met in Week 3, Part 4: name it.

In [ ]:
# Your code here

---

## 7. Recap

- An $8 \times 8$ image is a 64-feature row vector; dividing by the maximum brightness scales
  every feature to $[0, 1]$.
- Evaluate on held-out data. The test set is used only to judge, never to train or to choose.
- The lecture's $64 \to 25 \to 15 \to 1$ network has `2031` parameters, and separates 0s from 1s
  essentially perfectly.
- `model.predict` wants a batch: reshape one image to `(1, 64)`.
- Your NumPy forward propagation with the trained weights reproduces TensorFlow's outputs to
  float32 precision &mdash; looped or vectorised. There is no hidden step.
- First-layer weights can be viewed as images. On 0 vs 1, most units learn a version of one
  template: the difference between the average 1 and the average 0.

### Exercises to hand in

Hand in this notebook with both checkpoints completed and the following, each answered in a
markdown cell beneath your code.

1. **Harder pairs.** Extend checkpoint 2 to the pairs 3 vs 8, 1 vs 8 and 5 vs 9. Tabulate
   training and test accuracy for each, and explain which pair is hardest using the class-mean
   images.
2. **Width.** For 8 vs 9, retrain with the first layer reduced to 10, 5 and 2 units (keep the
   rest). Plot test accuracy against width. What is the smallest network that does as well as
   the original, and how many parameters does it have?
3. **Is depth needed?** Train a single sigmoid neuron (logistic regression, $64 \to 1$) on
   8 vs 9 with the same split, and compare its test accuracy with the network's. Relate the
   result to the coffee-roasting lab: what does it suggest about the shape of the boundary
   between 8s and 9s in 64-dimensional space, and about how far a 2031-parameter model fitted
   to 276 images can be trusted?
4. **Robustness.** Add Gaussian noise with standard deviation 0.1, 0.2 and 0.4 to the 0-vs-1
   test images (clip to $[0, 1]$). Report test accuracy at each level, and show the
   least-confident image at the highest level.
5. **Explain a prediction.** Choose one correctly classified test image. Multiply it, pixel by
   pixel, with the weights of the first-layer unit that best separates the classes (section 6),
   display the product as an image, and explain in two or three sentences what it shows about
   why that unit fired.

### Next

**Week 5 &mdash; Neural network training:** what `compile` and `fit` actually do, activation
functions other than the sigmoid, multiclass classification with softmax, and back-propagation.